In [1]:
import os
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr


In [2]:
load_dotenv(override=True)
openai = OpenAI()

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [3]:
def push(message):
    print(f"Push: {message}")
    payload = {
        "token": pushover_token,
        "user": pushover_user,
        "message": message
    }
    response = requests.post(pushover_url, data=payload)
    if response.status_code == 200:
        print("Push notification sent successfully.")
    else:
        print("Failed to send push notification.")

In [4]:
push("Hello from Hongduk's notebook!")

Push: Hello from Hongduk's notebook!
Push notification sent successfully.


In [5]:
def record_user_details(email, name="Name not provided", notes="Notes not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

def record_unknown_question(question):
    push(f"Recording {question} that I could not answer")
    return {"recorded": "ok"}

In [6]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool when the user is interested in getting in touch. The user will provide their email and optionally their name and additional notes",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The user's email address"
            },
            "name": {
                "type": "string",
                "description": "The user's name"
            },
            "notes": {
                "type": "string",
                "description": "Any message or additional information provided by user"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record ANY question that couldn't be answered because the information is not available in \
     the LinkedIn profile or summary. This includes ALL personal questions such as food preferences, hobbies, family details, daily routines, \
     or anything not documented professionally. NEVER guess or make up answers.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [7]:
tools = [{"type": "function", "function": record_user_details_json},
         {"type": "function", "function": record_unknown_question_json}]

In [8]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool when the user is interested in getting in touch. The user will provide their email and optionally their name and additional notes',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': "The user's email address"},
     'name': {'type': 'string', 'description': "The user's name"},
     'notes': {'type': 'string',
      'description': 'Any message or additional information provided by user'}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record ANY question that couldn't be answered because the information is not available in      the LinkedIn profile or summary. This includes ALL personal questions such as food preferences, hobbies, family details, daily routines,      or anything not documented professio

In [9]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {"error": f"Tool {tool_name} not found"}   
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [10]:
reader = PdfReader("me/linkedin_HD.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text 

with open("me/summary_HD.txt", "r", encoding="utf-8") as f:
    summary = f.read()

print(linkedin[:500])
print(summary)

   
연락처
mhda80@gmail.com
www.linkedin.com/in/hongduk-
ahn-9a1761149 (LinkedIn)
대표 보유기술
financial analysis
Project modeling
경영 컨설팅
Languages
English (Full Professional)
Certifications
Microsoft Certified: Azure
Fundamentals
Microsoft Certified: Azure Data
Fundamentals
Microsoft Certified: Security,
Compliance, and Identity
Fundamentals
Microsoft Certified: Azure AI
Fundamentals
데이터분석 준전문가(ADsP)
Patents
배터리 교환 장치 및 배터리 교환 시
스템
서버 및 그것의 동작 방법, 배터리 교
환 시스템
Hongduk Ahn
AI Adoption Architect
대한민국 서울
간
Hongduk is an AI Adoption Architect at Microsoft Korea, placed through LTIMindtree. He has 16+ years of enterprise experience at LG Group (LG CNS and LG Energy Solution) and 
holds an MBA in Finance from Korea University. He holds two PCT patents related to battery exchange systems. He is currently studying LLM Engineering while pursing to become an AI Enginner that can navigate/bridge the business and technical sector. 


In [11]:
name = "Hongduk Ahn"

system_prompt = f"""
You are acting as {name}. You are answering questions related to {name}'s career, background, skills and experience. 
Your responsibility is to represent {name} for interactions on the website, faithfully and professionally. 
Always be engaging and in character. Keep responses concise — no more than 2-3 sentences for casual greetings. 
Only give detailed responses when the user asks specific questions. 
If asked about ANYTHING not explicitly mentioned in your LinkedIn or summary — including personal preferences like food, hobbies, 
sports, family, daily life — you MUST use the record_unknown_question tool immediately. NEVER invent or guess personal details. 
This is mandatory — no exceptions.
If asked a question that you don't know the answer to, always use the record_unknown_question tool to record the question and tell the user you don't have that information.
If the user continues to be engaged and interested in the conversation, try to steer them to get in touch via email. But don't ask for the user's email in the first 3 messages. If they are interested in further discussion, ask for their email and record it using the record_user_details tool.
"""

system_prompt += f"\n\n## LinkedIn Profile:\n{linkedin}\n\n## Summary:\n{summary}\n\n"

In [12]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5-nano", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Push: Recording what is your favorite food? that I could not answer
Push notification sent successfully.
